# Advanced `UserDict` Problems — Tutorial Style

In this notebook we are going to continue working with `UserDict`, but this time the problems are deliberately developed in a **step-by-step tutorial style**.

Rather than immediately jumping to a finished class, we will:

- start with a requirement,
- build the smallest useful version,
- test one behavior,
- stop and inspect what happened,
- ask what happens with another dictionary method,
- discover edge cases,
- and gradually improve the design.

The problems are independent from the previous advanced-problems notebook.

We will focus on new custom mapping designs:

1. alias-aware dictionaries,
2. multi-value dictionaries,
3. bidirectional dictionaries,
4. lazy-loading dictionaries,
5. layered settings,
6. expiring dictionaries,
7. dictionaries with a global quota invariant,
8. observable dictionaries with callbacks,
9. and a capstone dependency-aware configuration dictionary.


### A small reminder

A `UserDict` object stores an ordinary dictionary in its `data` attribute.

The useful part for us is that the normal mapping machinery is implemented at the Python level, so overriding methods such as `__getitem__`, `__setitem__`, `__delitem__`, and `__iter__` gives us a natural way to customize dictionary behavior.

Let's import what we need.


In [1]:
from collections import UserDict
from collections.abc import Mapping
import math


We will also use a tiny helper when we deliberately want to demonstrate an exception.

That way, the notebook can still run from top to bottom without stopping.


In [2]:
def show_exception(exc_type, func, *args, **kwargs):
    try:
        func(*args, **kwargs)
    except exc_type as exc:
        print(f"{exc_type.__name__}: {exc}")
        return exc
    else:
        raise AssertionError(f"Expected {exc_type.__name__}")


# Problem 1 — An Alias-Aware Dictionary

Suppose we are storing information about a person.

Internally, we want to use these canonical keys:

```python
first_name
last_name
email
```

But we also want callers to be able to use shorter aliases:

```python
first -> first_name
last  -> last_name
mail  -> email
```

So these should refer to the same stored value:

```python
d['first']
d['first_name']
```

Let's build this gradually.


## Step 1 — Start with key normalization

The first thing we need is a way of converting any alias into its canonical key.

We could write that logic directly inside `__getitem__`, but we are also going to need it inside `__setitem__` and `__delitem__`.

So let's keep that operation in one helper method.


In [3]:
class AliasDict(UserDict):
    def __init__(self, aliases, *args, **kwargs):
        self._aliases = dict(aliases)
        super().__init__(*args, **kwargs)

    def _canonical_key(self, key):
        return self._aliases.get(key, key)

    def __getitem__(self, key):
        return super().__getitem__(self._canonical_key(key))

    def __setitem__(self, key, value):
        super().__setitem__(self._canonical_key(key), value)

    def __delitem__(self, key):
        super().__delitem__(self._canonical_key(key))


Let's try it.


In [4]:
aliases = {
    "first": "first_name",
    "last": "last_name",
    "mail": "email",
}

person = AliasDict(
    aliases,
    first="Ada",
    last_name="Lovelace",
    mail="ada@example.com",
)

person


{'first_name': 'Ada', 'last_name': 'Lovelace', 'email': 'ada@example.com'}

Interesting.

The representation shows the **canonical** keys.

That's reasonable because those are the keys actually stored in the underlying mapping.

Now let's see whether both spellings work when retrieving values.


In [5]:
person["first"], person["first_name"], person["mail"], person["email"]


('Ada', 'Ada', 'ada@example.com', 'ada@example.com')

Good.

But direct indexing is only one dictionary operation.

What about `get`?


In [6]:
person.get("first"), person.get("mail")


(None, None)

That works too.

Since `UserDict`'s inherited mapping behavior uses our customized lookup logic, `get` sees the aliases naturally.

What about assigning through an alias?


In [7]:
person["first"] = "Augusta Ada"
person


{'first_name': 'Augusta Ada', 'last_name': 'Lovelace', 'email': 'ada@example.com'}

Again, the alias did not become a second key.

It was normalized to `first_name`, which is exactly what we want.

But now we need to think about the alias table itself.

What if somebody creates an alias that points to another alias?


In [8]:
bad_aliases = {
    "first": "fname",
    "fname": "first_name",
}

bad = AliasDict(bad_aliases)
bad["first"] = "Ada"

bad.data


{'fname': 'Ada'}

Hmm...

Our current implementation performs only **one** alias lookup.

So `"first"` becomes `"fname"`, but it never continues to `"first_name"`.

We have two reasonable choices:

1. support alias chains, or
2. reject alias chains and require every alias to point directly to a canonical key.

For a small configuration object, the second rule is often easier to reason about.

Let's enforce it.


## Step 2 — Validate the alias configuration

We will reject:

- aliases that point to other aliases,
- aliases that map to themselves.

This makes normalization predictable.


In [9]:
class StrictAliasDict(UserDict):
    def __init__(self, aliases, *args, **kwargs):
        aliases = dict(aliases)

        for alias, canonical in aliases.items():
            if alias == canonical:
                raise ValueError(f"alias {alias!r} maps to itself")
            if canonical in aliases:
                raise ValueError(
                    f"alias {alias!r} points to another alias {canonical!r}"
                )

        self._aliases = aliases
        super().__init__(*args, **kwargs)

    def _canonical_key(self, key):
        return self._aliases.get(key, key)

    def __getitem__(self, key):
        return super().__getitem__(self._canonical_key(key))

    def __setitem__(self, key, value):
        super().__setitem__(self._canonical_key(key), value)

    def __delitem__(self, key):
        super().__delitem__(self._canonical_key(key))


In [10]:
show_exception(
    ValueError,
    StrictAliasDict,
    {"first": "fname", "fname": "first_name"},
)


ValueError: alias 'first' points to another alias 'fname'


ValueError("alias 'first' points to another alias 'fname'")

Now let's test some inherited methods.

Since deletion uses our canonicalization, `pop` should also cooperate with aliases.


In [11]:
person2 = StrictAliasDict(
    aliases,
    first="Ada",
    last="Lovelace",
    mail="ada@example.com",
)

removed = person2.pop("mail")

removed, person2


('ada@example.com', {'first_name': 'Ada', 'last_name': 'Lovelace'})

Excellent.

### Final observations

This problem illustrates a useful design pattern:

> If several mapping operations need the same transformation, put that transformation in a small helper and make the core special methods call it.

That keeps alias behavior consistent across direct lookup, assignment, `get`, `pop`, and other inherited mapping methods.


# Problem 2 — A Multi-Value Dictionary

Normal dictionaries store one value per key.

But HTTP headers, query parameters, form fields, and tags sometimes need **multiple values for the same key**.

Suppose we want this:

```python
d['color'] = 'red'
d['color'] = 'blue'
```

to store both values instead of replacing the first one.

Let's build that.


## Step 1 — Store a list for every key

The simplest approach is:

- if the key is new, create a one-element list,
- if the key already exists, append to the existing list.


In [12]:
class MultiValueDict(UserDict):
    def __setitem__(self, key, value):
        if key in self.data:
            self.data[key].append(value)
        else:
            self.data[key] = [value]


In [13]:
multi = MultiValueDict()

multi["color"] = "red"
multi["color"] = "blue"
multi["size"] = "large"

multi


{'color': ['red', 'blue'], 'size': ['large']}

That seems to work.

But what should normal indexing return?

At the moment it returns the whole list, because we have not customized `__getitem__`.

In many multi-value mapping APIs, normal indexing returns the most recent value, while a separate method returns all values.

Let's implement that.


## Step 2 — Separate the common case from the full history

We will define:

```python
d[key]        -> most recent value
d.getall(key) -> all values
```


In [14]:
class MultiValueDict(UserDict):
    def __setitem__(self, key, value):
        if key in self.data:
            self.data[key].append(value)
        else:
            self.data[key] = [value]

    def __getitem__(self, key):
        values = self.data[key]
        return values[-1]

    def getall(self, key):
        return list(self.data[key])


In [15]:
multi = MultiValueDict()
multi["color"] = "red"
multi["color"] = "blue"
multi["color"] = "green"

multi["color"], multi.getall("color"), multi.data


('green', ['red', 'blue', 'green'], {'color': ['red', 'blue', 'green']})

Notice that `getall` returns a **copy** of the list.

Why bother?

Because if we returned the actual internal list, code outside the class could mutate it without going through our mapping interface.


In [16]:
colors = multi.getall("color")
colors.append("purple")

print("Returned list:", colors)
print("Stored list:", multi.data["color"])


Returned list: ['red', 'blue', 'green', 'purple']
Stored list: ['red', 'blue', 'green']


Good — changing the returned list did not change our internal storage.

Now let's try something more interesting.

What happens if we initialize from a sequence containing duplicate keys?


In [17]:
multi2 = MultiValueDict([
    ("tag", "python"),
    ("tag", "collections"),
    ("tag", "mapping"),
])

multi2.data


{'tag': ['python', 'collections', 'mapping']}

Nice!

The initializer eventually routes the entries through our `__setitem__`, so the repeated keys are accumulated.

What about `update`?


In [18]:
multi2.update([
    ("tag", "userdict"),
    ("level", "advanced"),
])

multi2.data


{'tag': ['python', 'collections', 'mapping', 'userdict'],
 'level': ['advanced']}

That worked as well.

There is one more design issue.

What should this mean?

```python
multi2['tag'] = ['a', 'b']
```

Our current class treats that entire list as **one value**.

That may actually be correct, but if we want a special bulk API, it is better to make it explicit rather than guessing based on the input type.

Let's add `setall`.


## Step 3 — Add an explicit replacement operation


In [19]:
class MultiValueDict(UserDict):
    def __setitem__(self, key, value):
        if key in self.data:
            self.data[key].append(value)
        else:
            self.data[key] = [value]

    def __getitem__(self, key):
        return self.data[key][-1]

    def getall(self, key):
        return list(self.data[key])

    def setall(self, key, values):
        values = list(values)
        if not values:
            raise ValueError("values cannot be empty")
        self.data[key] = values


In [20]:
multi3 = MultiValueDict(tag="python")
multi3["tag"] = "userdict"
multi3.setall("tag", ["mapping", "collections"])

print(multi3["tag"])
print(multi3.getall("tag"))
print(multi3.data)


collections
['mapping', 'collections']
{'tag': ['mapping', 'collections']}


### Final observations

The important API-design lesson here is that a custom mapping does not have to make `__setitem__` do everything.

Sometimes an explicit method such as `getall` or `setall` makes the behavior clearer and avoids ambiguous magic.


# Problem 3 — A Bidirectional Dictionary

Suppose we want a mapping where both keys **and values** are unique.

For example:

```python
country_code['Bulgaria'] == 'BG'
```

and we also want:

```python
country_code.key_for('BG') == 'Bulgaria'
```

That means we need to maintain two relationships:

```text
key -> value
value -> key
```

Let's build this carefully.


## Step 1 — Maintain an inverse dictionary

We will keep the normal mapping in `data`, and a second dictionary called `_inverse`.


In [21]:
class BiDict(UserDict):
    def __init__(self, *args, **kwargs):
        self._inverse = {}
        super().__init__(*args, **kwargs)

    def __setitem__(self, key, value):
        if value in self._inverse:
            raise ValueError(f"value {value!r} is already in use")

        super().__setitem__(key, value)
        self._inverse[value] = key

    def key_for(self, value):
        return self._inverse[value]


In [22]:
codes = BiDict()
codes["Bulgaria"] = "BG"
codes["France"] = "FR"

codes, codes.key_for("BG")


({'Bulgaria': 'BG', 'France': 'FR'}, 'Bulgaria')

The uniqueness check works for two different keys.

Let's confirm.


In [23]:
show_exception(
    ValueError,
    codes.__setitem__,
    "Other",
    "BG",
)


ValueError: value 'BG' is already in use


ValueError("value 'BG' is already in use")

So far so good.

Now let's try **changing** an existing key.


In [24]:
codes["Bulgaria"] = "BGR"

print(codes)
print(codes._inverse)

assert codes["Bulgaria"] == "BGR"
assert codes._inverse["BG"] == "Bulgaria"  # stale inverse entry: the bug


{'Bulgaria': 'BGR', 'France': 'FR'}
{'BG': 'Bulgaria', 'FR': 'France', 'BGR': 'Bulgaria'}


We found a bug.

Why?

When `"Bulgaria"` already exists, the old inverse relationship:

```python
'BG' -> 'Bulgaria'
```

must be removed before we replace the value.

Our first implementation never handled reassignment.

Let's fix that.


## Step 2 — Handle reassignment correctly

We need to consider three cases:

1. new key + new value,
2. existing key + same value,
3. existing key + different unused value.

And of course we still reject a value owned by some **other** key.


In [25]:
class BiDict(UserDict):
    def __init__(self, *args, **kwargs):
        self._inverse = {}
        super().__init__(*args, **kwargs)

    def __setitem__(self, key, value):
        current_owner = self._inverse.get(value)

        if current_owner is not None and current_owner != key:
            raise ValueError(f"value {value!r} is already in use")

        if key in self.data:
            old_value = self.data[key]
            if old_value == value:
                return
            del self._inverse[old_value]

        self.data[key] = value
        self._inverse[value] = key

    def __delitem__(self, key):
        value = self.data[key]
        del self.data[key]
        del self._inverse[value]

    def key_for(self, value):
        return self._inverse[value]


In [26]:
codes = BiDict(
    Bulgaria="BG",
    France="FR",
)

codes["Bulgaria"] = "BGR"

print(codes)
print(codes._inverse)

assert codes.key_for("BGR") == "Bulgaria"
assert "BG" not in codes._inverse


{'Bulgaria': 'BGR', 'France': 'FR'}
{'FR': 'France', 'BGR': 'Bulgaria'}


What about deletion?

If we delete a forward entry, the inverse entry must disappear too.


In [27]:
del codes["France"]

print(codes)
print(codes._inverse)

assert "FR" not in codes._inverse


{'Bulgaria': 'BGR'}
{'BGR': 'Bulgaria'}


Now let's test initialization with duplicate values.


In [28]:
show_exception(
    ValueError,
    BiDict,
    [("a", 1), ("b", 1)],
)


ValueError: value 1 is already in use


ValueError('value 1 is already in use')

Excellent.

Because initialization uses our assignment behavior, the uniqueness invariant is enforced there too.

### Final observations

This problem is more subtle than value validation.

We are maintaining a **relationship between two pieces of state**.

Whenever the forward mapping changes, the inverse mapping must change in exactly the corresponding way.


# Problem 4 — A Lazy-Loading Dictionary

Suppose some values are expensive to compute.

We do not want to load them until the key is actually requested.

For example:

```python
cache['customer:42']
```

should call a loader function only if that key is missing.

This is a good use case for `__missing__`.


## Step 1 — A loader-backed mapping

We will pass a function to the constructor.

When a missing key is requested:

1. call the loader,
2. store the result,
3. return it.


In [29]:
class LazyDict(UserDict):
    def __init__(self, loader, *args, **kwargs):
        self._loader = loader
        super().__init__(*args, **kwargs)

    def __missing__(self, key):
        value = self._loader(key)
        self[key] = value
        return value


Let's make a loader that tells us when it is called.


In [30]:
load_calls = []

def loader(key):
    load_calls.append(key)
    return key.upper()

lazy = LazyDict(loader)


The dictionary is empty and the loader has not been called yet.


In [31]:
lazy.data, load_calls


({}, [])

Now let's request a missing key.


In [32]:
lazy["python"]


'PYTHON'

In [33]:
lazy.data, load_calls


({'python': 'PYTHON'}, ['python'])

As expected, the value was loaded and cached.

What happens if we request the same key again?


In [34]:
lazy["python"]
lazy["python"]

load_calls


['python']

Only one load occurred.

That's the basic cache behavior we wanted.

Now what about `get`?


In [35]:
lazy.get("collections"), load_calls


(None, ['python'])

Interesting and useful: inherited `get` also goes through our lookup behavior, so it can trigger `__missing__`.

But sometimes we want a lookup that **does not** load anything.

Let's add one.


## Step 2 — Add a non-loading lookup

We will call it `peek`.

It should inspect only currently cached values.


In [36]:
class LazyDict(UserDict):
    def __init__(self, loader, *args, **kwargs):
        self._loader = loader
        super().__init__(*args, **kwargs)

    def __missing__(self, key):
        value = self._loader(key)
        self[key] = value
        return value

    def peek(self, key, default=None):
        return self.data.get(key, default)


In [37]:
load_calls = []
lazy = LazyDict(loader)

print(lazy.peek("abc", "NOT CACHED"))
print(load_calls)

print(lazy["abc"])
print(load_calls)

print(lazy.peek("abc"))


NOT CACHED
[]
ABC
['abc']
ABC


Now let's add one more realistic feature.

Suppose the loader may return `None` when the key does not exist.

We do **not** want to cache that as though it were a real value.

Let's reject it.


## Step 3 — Validate loaded values


In [38]:
class LazyDict(UserDict):
    def __init__(self, loader, *args, **kwargs):
        self._loader = loader
        super().__init__(*args, **kwargs)

    def __missing__(self, key):
        value = self._loader(key)

        if value is None:
            raise KeyError(key)

        self[key] = value
        return value

    def peek(self, key, default=None):
        return self.data.get(key, default)


In [39]:
def selective_loader(key):
    if key == "known":
        return 100
    return None

lazy2 = LazyDict(selective_loader)

print(lazy2["known"])
show_exception(KeyError, lazy2.__getitem__, "unknown")
print(lazy2.data)


100
KeyError: 'unknown'
{'known': 100}


### Final observations

A custom mapping can distinguish among:

- a normal cached lookup,
- a lookup that triggers loading,
- and a lookup that only inspects the cache.

That is much clearer than trying to overload one method with too many meanings.


# Problem 5 — Layered Settings

Suppose an application has:

- default settings,
- user overrides.

We want one mapping interface that reads from the override first, then falls back to the defaults.

For example:

```python
defaults = {'theme': 'light', 'language': 'en'}
overrides = {'theme': 'dark'}
```

The public mapping should appear to contain both keys, but only the override should be stored in `data`.

This one requires us to think about more than just `__getitem__`.


## Step 1 — Implement fallback lookup

Let's start with the obvious part.


In [40]:
class LayeredDict(UserDict):
    def __init__(self, defaults, *args, **kwargs):
        self._defaults = dict(defaults)
        super().__init__(*args, **kwargs)

    def __getitem__(self, key):
        if key in self.data:
            return self.data[key]
        return self._defaults[key]


In [41]:
settings = LayeredDict(
    {"theme": "light", "language": "en", "timeout": 30},
    theme="dark",
)

settings["theme"], settings["language"], settings.data


('dark', 'en', {'theme': 'dark'})

Lookup works.

But let's inspect the mapping itself.


In [42]:
list(settings), len(settings), dict(settings)


(['theme'], 1, {'theme': 'dark'})

That's not right.

The inherited iteration only sees the keys in `data`, so the default-only keys are invisible.

If we want this object to behave like a true merged mapping, we also need to customize iteration and length.


## Step 2 — Make iteration represent the public mapping


In [43]:
class LayeredDict(UserDict):
    def __init__(self, defaults, *args, **kwargs):
        self._defaults = dict(defaults)
        super().__init__(*args, **kwargs)

    def __getitem__(self, key):
        if key in self.data:
            return self.data[key]
        return self._defaults[key]

    def __contains__(self, key):
        return key in self.data or key in self._defaults


    def __iter__(self):
        seen = set()

        for key in self.data:
            seen.add(key)
            yield key

        for key in self._defaults:
            if key not in seen:
                yield key

    def __len__(self):
        return len(set(self.data) | set(self._defaults))


In [44]:
settings = LayeredDict(
    {"theme": "light", "language": "en", "timeout": 30},
    theme="dark",
)

print(list(settings))
print(len(settings))
print(dict(settings))


['theme', 'language', 'timeout']
3
{'theme': 'dark', 'language': 'en', 'timeout': 30}


Much better.

Now let's think about deletion.

Suppose we do:

```python
del settings['theme']
```

Do we want the key to disappear completely?

Or do we want to remove only the override so that the default becomes visible again?

For layered settings, the second behavior is usually more useful.

There is one more subtle part of the public mapping: membership. `UserDict.get()` checks whether a key is contained before retrieving it, so default-only keys should also be recognized by `__contains__`.


## Step 3 — Deleting an override reveals the default


In [45]:
class LayeredDict(UserDict):
    def __init__(self, defaults, *args, **kwargs):
        self._defaults = dict(defaults)
        super().__init__(*args, **kwargs)

    def __getitem__(self, key):
        if key in self.data:
            return self.data[key]
        return self._defaults[key]

    def __contains__(self, key):
        return key in self.data or key in self._defaults


    def __iter__(self):
        seen = set()
        for key in self.data:
            seen.add(key)
            yield key
        for key in self._defaults:
            if key not in seen:
                yield key

    def __len__(self):
        return len(set(self.data) | set(self._defaults))

    def __delitem__(self, key):
        if key in self.data:
            del self.data[key]
            return

        if key in self._defaults:
            raise KeyError(
                f"{key!r} is a default-only key; there is no override to delete"
            )

        raise KeyError(key)


In [46]:
settings = LayeredDict(
    {"theme": "light", "language": "en"},
    theme="dark",
)

print(settings["theme"])

del settings["theme"]

print(settings["theme"])
print(settings.data)


dark
light
{}


Exactly what we wanted: deleting the override exposed the default.

What happens if we try deleting a default-only key?


In [47]:
show_exception(KeyError, settings.__delitem__, "language")


KeyError: "'language' is a default-only key; there is no override to delete"


KeyError("'language' is a default-only key; there is no override to delete")

### Final observations

This problem shows that once the public mapping is conceptually different from `data`, you need to think about the full mapping protocol:

- lookup,
- iteration,
- length,
- deletion,
- conversion to `dict`,
- and inherited methods.

Overriding only `__getitem__` would have produced an inconsistent object.


# Problem 6 — An Expiring Dictionary

Now let's make a dictionary whose entries expire after a fixed amount of time.

We could use the real clock, but that would make the tests slow and unreliable.

Instead, we'll inject a clock function.

That gives us deterministic tests.


## Step 1 — Build a fake clock

Our fake clock simply stores a number and lets us advance it manually.


In [48]:
class FakeClock:
    def __init__(self):
        self.now = 0.0

    def __call__(self):
        return self.now

    def advance(self, seconds):
        self.now += seconds


In [49]:
clock = FakeClock()

print(clock())
clock.advance(5)
print(clock())


0.0
5.0


Good.

Now each dictionary entry needs two pieces of information:

```text
value
expiry time
```

We'll keep those together internally.


## Step 2 — Store `(value, expires_at)` internally


In [50]:
class ExpiringDict(UserDict):
    def __init__(self, ttl, clock, *args, **kwargs):
        if ttl <= 0:
            raise ValueError("ttl must be positive")

        self._ttl = float(ttl)
        self._clock = clock
        super().__init__(*args, **kwargs)

    def __setitem__(self, key, value):
        expires_at = self._clock() + self._ttl
        self.data[key] = (value, expires_at)

    def __getitem__(self, key):
        value, expires_at = self.data[key]

        if self._clock() >= expires_at:
            del self.data[key]
            raise KeyError(key)

        return value


In [51]:
clock = FakeClock()
cache = ExpiringDict(10, clock)

cache["token"] = "abc"

print(cache["token"])
print(cache.data)


abc
{'token': ('abc', 10.0)}


The internal representation contains the expiry timestamp, while public lookup returns only the value.

Let's advance time, but not enough for expiration.


In [52]:
clock.advance(9)
cache["token"]


'abc'

Still there.

Now one more second.


In [53]:
clock.advance(1)

show_exception(KeyError, cache.__getitem__, "token")
print(cache.data)


KeyError: 'token'
{}


The expired item was removed lazily when we tried to read it.

But there is a consistency problem.

What does `len(cache)` report if an entry has expired but nobody has tried to read it yet?


In [54]:
clock = FakeClock()
cache = ExpiringDict(5, clock)

cache["a"] = 1
cache["b"] = 2

clock.advance(10)

print("len(cache):", len(cache))
print("raw data:", cache.data)


len(cache): 2
raw data: {'a': (1, 5.0), 'b': (2, 5.0)}


The raw entries are still present, so inherited length counts them.

If the public meaning of the mapping is "only non-expired entries", we should clean expired entries before iteration and length operations.


## Step 3 — Purge expired entries for mapping-wide operations


In [55]:
class ExpiringDict(UserDict):
    def __init__(self, ttl, clock, *args, **kwargs):
        if ttl <= 0:
            raise ValueError("ttl must be positive")

        self._ttl = float(ttl)
        self._clock = clock
        super().__init__(*args, **kwargs)

    def __setitem__(self, key, value):
        self.data[key] = (value, self._clock() + self._ttl)

    def __getitem__(self, key):
        value, expires_at = self.data[key]

        if self._clock() >= expires_at:
            del self.data[key]
            raise KeyError(key)

        return value

    def purge_expired(self):
        now = self._clock()
        expired = [
            key
            for key, (_, expires_at) in self.data.items()
            if now >= expires_at
        ]

        for key in expired:
            del self.data[key]

        return len(expired)

    def __iter__(self):
        self.purge_expired()
        return iter(self.data)

    def __len__(self):
        self.purge_expired()
        return len(self.data)


In [56]:
clock = FakeClock()
cache = ExpiringDict(5, clock)

cache["a"] = 1
cache["b"] = 2

clock.advance(10)

print("len(cache):", len(cache))
print("raw data:", cache.data)


len(cache): 0
raw data: {}


Now the public mapping and internal state agree after a mapping-wide operation.

### Final observations

Injecting dependencies such as a clock makes behavior easier to test.

It also makes the class less tightly coupled to global state.


# Problem 7 — A Dictionary With a Global Quota

So far most validation rules have applied to one item at a time.

Let's make a harder invariant.

Suppose our dictionary allocates a total resource budget of 100 units.

These assignments are valid:

```python
cpu = 40
memory = 30
disk = 20
```

because the total is 90.

But another 20-unit allocation must be rejected because the total would become 110.

The important part is that the validity of one assignment depends on **all the other values**.


## Step 1 — Validate a proposed assignment against the total

When replacing an existing key, we must subtract its old value before checking the proposed total.


In [57]:
class QuotaDict(UserDict):
    def __init__(self, capacity, *args, **kwargs):
        if capacity < 0:
            raise ValueError("capacity cannot be negative")

        self._capacity = capacity
        super().__init__(*args, **kwargs)

    def __setitem__(self, key, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("allocation must be numeric")

        if not math.isfinite(value) or value < 0:
            raise ValueError("allocation must be finite and non-negative")

        old_value = self.data.get(key, 0)
        proposed_total = sum(self.data.values()) - old_value + value

        if proposed_total > self._capacity:
            raise ValueError(
                f"capacity exceeded: {proposed_total} > {self._capacity}"
            )

        super().__setitem__(key, value)

    @property
    def used(self):
        return sum(self.data.values())

    @property
    def remaining(self):
        return self._capacity - self.used


In [58]:
quota = QuotaDict(100)

quota["cpu"] = 40
quota["memory"] = 30
quota["disk"] = 20

print(quota)
print("used:", quota.used)
print("remaining:", quota.remaining)


{'cpu': 40, 'memory': 30, 'disk': 20}
used: 90
remaining: 10


Now let's try an allocation that does not fit.


In [59]:
show_exception(
    ValueError,
    quota.__setitem__,
    "network",
    20,
)

print(quota)


ValueError: capacity exceeded: 110 > 100
{'cpu': 40, 'memory': 30, 'disk': 20}


Good.

What about replacing an existing allocation with a smaller value?


In [60]:
quota["cpu"] = 25

print(quota)
print("remaining:", quota.remaining)


{'cpu': 25, 'memory': 30, 'disk': 20}
remaining: 25


That works because we removed the old CPU allocation from the proposed total before adding the new one.

Now let's try a bulk update.


In [61]:
quota2 = QuotaDict(100, a=20, b=20)

print("Before:", quota2)

show_exception(
    ValueError,
    quota2.update,
    {"c": 50, "d": 30},
)

print("After:", quota2)


Before: {'a': 20, 'b': 20}
ValueError: capacity exceeded: 120 > 100
After: {'a': 20, 'b': 20, 'c': 50}


Aha!

We have discovered an important issue.

The update attempted the entries one at a time:

- `c = 50` was valid, so it was stored.
- `d = 30` then pushed the total over capacity and failed.

So the update raised an error **after partially changing the dictionary**.

For some applications that is acceptable.

For resource allocation, it is usually not.

Let's make bulk updates all-or-nothing.


## Step 2 — Validate the complete candidate state before committing

We will create a temporary plain dictionary representing the proposed final state.

If that state is valid, we commit it.

If it is invalid, the original object remains unchanged.


In [62]:
class QuotaDict(UserDict):
    def __init__(self, capacity, *args, **kwargs):
        if capacity < 0:
            raise ValueError("capacity cannot be negative")

        self._capacity = capacity
        super().__init__()

        if args or kwargs:
            self.update(*args, **kwargs)

    def _validate_value(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("allocation must be numeric")
        if not math.isfinite(value) or value < 0:
            raise ValueError("allocation must be finite and non-negative")

    def _validate_state(self, state):
        for value in state.values():
            self._validate_value(value)

        total = sum(state.values())
        if total > self._capacity:
            raise ValueError(
                f"capacity exceeded: {total} > {self._capacity}"
            )

    def __setitem__(self, key, value):
        candidate = dict(self.data)
        candidate[key] = value
        self._validate_state(candidate)
        self.data[key] = value

    def update(self, other=None, **kwargs):
        candidate = dict(self.data)

        if other is not None:
            if isinstance(other, Mapping) or hasattr(other, "keys"):
                for key in other.keys():
                    candidate[key] = other[key]
            else:
                for key, value in other:
                    candidate[key] = value

        candidate.update(kwargs)

        self._validate_state(candidate)
        self.data = candidate

    @property
    def used(self):
        return sum(self.data.values())

    @property
    def remaining(self):
        return self._capacity - self.used


In [63]:
quota3 = QuotaDict(100, a=20, b=20)

before = dict(quota3)

show_exception(
    ValueError,
    quota3.update,
    {"c": 50, "d": 30},
)

print("Before:", before)
print("After:", quota3)

assert dict(quota3) == before


ValueError: capacity exceeded: 120 > 100
Before: {'a': 20, 'b': 20}
After: {'a': 20, 'b': 20}


Much better.

### Final observations

This is a different kind of invariant:

> The validity of one entry depends on the state of the entire mapping.

Whenever you have a global invariant, bulk operations deserve special attention.


# Problem 8 — An Observable Dictionary

Sometimes we want code to react whenever a mapping changes.

For example:

- refresh a UI,
- invalidate another cache,
- update a derived object,
- send a metric,
- or trigger application logic.

Let's make a dictionary that calls subscribed functions after successful changes.


## Step 1 — Store listeners

A listener will receive:

```python
event
key
old_value
new_value
```

Let's use a sentinel object when there was no previous value.


In [64]:
_NOT_PRESENT = object()

class ObservableDict(UserDict):
    def __init__(self, *args, **kwargs):
        self._listeners = []
        super().__init__(*args, **kwargs)

    def subscribe(self, callback):
        if callback not in self._listeners:
            self._listeners.append(callback)

    def unsubscribe(self, callback):
        self._listeners.remove(callback)

    def _notify(self, event, key, old_value, new_value):
        for callback in tuple(self._listeners):
            callback(event, key, old_value, new_value)

    def __setitem__(self, key, value):
        old_value = self.data.get(key, _NOT_PRESENT)
        super().__setitem__(key, value)
        self._notify("set", key, old_value, value)

    def __delitem__(self, key):
        old_value = self.data[key]
        super().__delitem__(key)
        self._notify("delete", key, old_value, _NOT_PRESENT)


Let's make a listener that records what it sees.


In [65]:
events = []

def listener(event, key, old_value, new_value):
    events.append((event, key, old_value, new_value))

observable = ObservableDict()
observable.subscribe(listener)

observable["x"] = 10
observable["x"] = 20
del observable["x"]

len(events)


3

Let's display the events in a friendlier form.


In [66]:
for event, key, old_value, new_value in events:
    old_display = "<missing>" if old_value is _NOT_PRESENT else old_value
    new_display = "<missing>" if new_value is _NOT_PRESENT else new_value
    print(event, key, old_display, "->", new_display)


set x <missing> -> 10
set x 10 -> 20
delete x 20 -> <missing>


That works.

But now think about initialization.

We used:

```python
super().__init__(*args, **kwargs)
```

and initialization itself may route through `__setitem__`.

At that moment, our listener list already exists, but no listeners have been subscribed yet.

So initial values will not notify anybody.

That is actually a sensible behavior.

Now let's make the class slightly more useful by avoiding notifications when a value does not really change.


## Step 2 — Suppress no-op changes


In [67]:
class ObservableDict(UserDict):
    def __init__(self, *args, **kwargs):
        self._listeners = []
        super().__init__(*args, **kwargs)

    def subscribe(self, callback):
        if callback not in self._listeners:
            self._listeners.append(callback)

    def unsubscribe(self, callback):
        self._listeners.remove(callback)

    def _notify(self, event, key, old_value, new_value):
        for callback in tuple(self._listeners):
            callback(event, key, old_value, new_value)

    def __setitem__(self, key, value):
        old_value = self.data.get(key, _NOT_PRESENT)

        if old_value is not _NOT_PRESENT and old_value == value:
            return

        super().__setitem__(key, value)
        self._notify("set", key, old_value, value)

    def __delitem__(self, key):
        old_value = self.data[key]
        super().__delitem__(key)
        self._notify("delete", key, old_value, _NOT_PRESENT)


In [68]:
events = []

observable = ObservableDict()
observable.subscribe(listener)

observable["mode"] = "dark"
observable["mode"] = "dark"
observable["mode"] = "light"

print("Event count:", len(events))

for event, key, old_value, new_value in events:
    old_display = "<missing>" if old_value is _NOT_PRESENT else old_value
    new_display = "<missing>" if new_value is _NOT_PRESENT else new_value
    print(event, key, old_display, "->", new_display)


Event count: 2
set mode <missing> -> dark
set mode dark -> light


Only the real changes produced notifications.

One more subtle point: we iterate over `tuple(self._listeners)` rather than the live list itself.

Why?

Because a callback might subscribe or unsubscribe a listener while notifications are being delivered.

Iterating over a snapshot prevents that from corrupting the loop.


### Final observations

This design demonstrates that `UserDict` can customize **side effects**, not only stored values.

But side effects make correctness more important:

- notify only after a successful change,
- define what counts as a change,
- and make callback iteration robust.


# Problem 9 — Capstone: Dependency-Aware Configuration

Let's combine several ideas into a more substantial problem.

Suppose we store configuration values where some values are **derived** from others.

We want this public mapping:

```python
config['width']
config['height']
config['area']
config['perimeter']
```

But callers are allowed to assign only:

```python
width
height
```

The keys `area` and `perimeter` are computed.

We also want ordinary mapping operations such as:

```python
get
keys
items
dict(config)
```

to include the derived values.

This requires us to distinguish carefully between:

- stored keys,
- public keys,
- writable keys,
- computed keys.


## Step 1 — Store only the writable base values

Let's start with `width` and `height`.

Both must be positive finite real numbers.


In [69]:
class RectangleConfig(UserDict):
    _writable = {"width", "height"}

    def __setitem__(self, key, value):
        if key not in self._writable:
            raise KeyError(f"{key!r} is not writable")

        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("dimensions must be numeric")

        if not math.isfinite(value) or value <= 0:
            raise ValueError("dimensions must be positive and finite")

        super().__setitem__(key, value)


In [70]:
rect = RectangleConfig(width=5, height=3)
rect


{'width': 5, 'height': 3}

So far, `area` and `perimeter` do not exist.

Let's add them as computed lookups.


## Step 2 — Compute derived keys during lookup


In [71]:
class RectangleConfig(UserDict):
    _writable = {"width", "height"}
    _derived = {"area", "perimeter"}

    def __setitem__(self, key, value):
        if key not in self._writable:
            raise KeyError(f"{key!r} is not writable")

        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("dimensions must be numeric")

        if not math.isfinite(value) or value <= 0:
            raise ValueError("dimensions must be positive and finite")

        super().__setitem__(key, value)

    def __getitem__(self, key):
        if key == "area":
            return self.data["width"] * self.data["height"]

        if key == "perimeter":
            return 2 * (self.data["width"] + self.data["height"])

        return super().__getitem__(key)

    def __contains__(self, key):
        return key in self.data or key in self._derived


In [72]:
rect = RectangleConfig(width=5, height=3)

rect["area"], rect["perimeter"]


(15, 16)

Good.

What about `get`?


In [73]:
rect.get("area"), rect.get("perimeter")


(15, 16)

That works.

But what about `keys()` and `dict(rect)`?


In [74]:
print(list(rect.keys()))
print(dict(rect))


['width', 'height']
{'width': 5, 'height': 3}


The derived keys are missing from iteration.

This is similar to the layered-settings problem.

If `area` and `perimeter` are part of the **public mapping**, then iteration and length need to know about them.

There is another subtle point here: `get()` first checks membership. Since derived keys are public keys even though they are not stored in `.data`, our class also needs `__contains__` to recognize them.


## Step 3 — Make derived values part of the public mapping


In [75]:
class RectangleConfig(UserDict):
    _writable = ("width", "height")
    _derived = ("area", "perimeter")

    def __setitem__(self, key, value):
        if key not in self._writable:
            raise KeyError(f"{key!r} is not writable")

        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("dimensions must be numeric")

        if not math.isfinite(value) or value <= 0:
            raise ValueError("dimensions must be positive and finite")

        self.data[key] = value

    def __getitem__(self, key):
        if key == "area":
            return self.data["width"] * self.data["height"]

        if key == "perimeter":
            return 2 * (self.data["width"] + self.data["height"])

        return self.data[key]

    def __contains__(self, key):
        if key in self.data:
            return True
        return key in self._derived and all(
            required in self.data for required in self._writable
        )


    def __iter__(self):
        for key in self._writable:
            if key in self.data:
                yield key

        if all(key in self.data for key in self._writable):
            yield from self._derived

    def __len__(self):
        return sum(1 for _ in self)


In [76]:
rect = RectangleConfig(width=5, height=3)

print(list(rect.keys()))
print(list(rect.items()))
print(dict(rect))
print(len(rect))


['width', 'height', 'area', 'perimeter']
[('width', 5), ('height', 3), ('area', 15), ('perimeter', 16)]
{'width': 5, 'height': 3, 'area': 15, 'perimeter': 16}
4


Now the derived values behave like proper public mapping entries.

Let's verify that they still cannot be assigned.


In [77]:
show_exception(
    KeyError,
    rect.__setitem__,
    "area",
    999,
)


KeyError: "'area' is not writable"


KeyError("'area' is not writable")

What about deletion?

Deleting a derived key does not make sense, because it is not stored.

Deleting `width` or `height` is more interesting: once one dimension is missing, `area` and `perimeter` should disappear from the public key set.

Let's see what the inherited deletion gives us.


In [78]:
del rect["width"]

print(rect.data)
print(list(rect))
print(dict(rect))


{'height': 3}
['height']
{'height': 3}


That actually gives us a coherent result:

- `width` is gone,
- `height` remains,
- the derived keys are no longer iterable because the required inputs are incomplete.

Now let's ask another question.

What if the object is supposed to represent a **complete rectangle** at all times?

Then allowing deletion would violate that design.

We can make the final version disallow deletion entirely once configured.


## Step 4 — Finalize the invariant

The final class will require both dimensions during construction and disallow deletion.

This makes every valid instance represent a complete rectangle.


In [79]:
class RectangleConfig(UserDict):
    _writable = ("width", "height")
    _derived = ("area", "perimeter")

    def __init__(self, width, height):
        super().__init__()
        self["width"] = width
        self["height"] = height

    def _validate_dimension(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("dimensions must be numeric")

        if not math.isfinite(value) or value <= 0:
            raise ValueError("dimensions must be positive and finite")

    def __setitem__(self, key, value):
        if key not in self._writable:
            raise KeyError(f"{key!r} is not writable")

        self._validate_dimension(value)
        self.data[key] = value

    def __getitem__(self, key):
        if key == "area":
            return self.data["width"] * self.data["height"]

        if key == "perimeter":
            return 2 * (self.data["width"] + self.data["height"])

        return self.data[key]

    def __contains__(self, key):
        if key in self.data:
            return True
        return key in self._derived and all(
            required in self.data for required in self._writable
        )


    def __iter__(self):
        yield from self._writable
        yield from self._derived

    def __len__(self):
        return len(self._writable) + len(self._derived)

    def __delitem__(self, key):
        raise TypeError("rectangle dimensions cannot be deleted")


In [80]:
rect = RectangleConfig(8, 2.5)

print(rect.data)
print(dict(rect))

assert rect["area"] == 20
assert rect["perimeter"] == 21


{'width': 8, 'height': 2.5}
{'width': 8, 'height': 2.5, 'area': 20.0, 'perimeter': 21.0}


Let's test inherited mapping methods one more time.


In [81]:
print(rect.get("area"))
print(list(rect.keys()))
print(list(rect.values()))
print(list(rect.items()))


20.0
['width', 'height', 'area', 'perimeter']
[8, 2.5, 20.0, 21.0]
[('width', 8), ('height', 2.5), ('area', 20.0), ('perimeter', 21.0)]


And let's verify the invalid cases.


In [82]:
show_exception(TypeError, RectangleConfig, True, 10)
show_exception(ValueError, RectangleConfig, -1, 10)
show_exception(KeyError, rect.__setitem__, "area", 100)
show_exception(TypeError, rect.__delitem__, "width")


TypeError: dimensions must be numeric
ValueError: dimensions must be positive and finite
KeyError: "'area' is not writable"
TypeError: rectangle dimensions cannot be deleted


TypeError('rectangle dimensions cannot be deleted')

### Capstone observations

This final problem illustrates a key idea that appears again and again in custom mapping design:

> The contents of `data` do not have to be identical to the public mapping interface.

Here:

- `data` stores only `width` and `height`,
- `area` and `perimeter` are public mapping keys,
- they are computed dynamically,
- they appear in `get`, `keys`, `values`, `items`, and `dict(rect)`,
- but they cannot be assigned or deleted.

Once we make that choice, we have to make the whole mapping protocol tell a consistent story.


# Additional Practice — Without Solutions First

Try these before reading the solution cells below.

## Challenge 1

Modify `MultiValueDict` so that:

```python
del d[key]
```

removes only the most recent value.

If that was the last value, remove the key completely.

---

## Challenge 2

Modify `BiDict` to support:

```python
value in d.values()
```

efficiently through a method called:

```python
has_value(value)
```

Do not scan the forward dictionary.

---

## Challenge 3

Add a `refresh(key)` method to `LazyDict`.

It should force the loader to run again even if the key is already cached.

---

## Challenge 4

Add a method to `LayeredDict`:

```python
is_overridden(key)
```

It should tell you whether the currently visible value comes from `data` rather than the defaults.

---

## Challenge 5

Add `touch(key)` to `ExpiringDict`.

It should reset the expiry timer without changing the value.


# Additional Practice — Solutions


## Challenge 1 Solution


In [83]:
class PopLastMultiValueDict(MultiValueDict):
    def __delitem__(self, key):
        values = self.data[key]
        values.pop()

        if not values:
            del self.data[key]


m = PopLastMultiValueDict()
m["x"] = 1
m["x"] = 2
m["x"] = 3

del m["x"]
assert m.getall("x") == [1, 2]

del m["x"]
del m["x"]

assert "x" not in m
print(m)


{}


The important detail is that we mutate the stored list first, then remove the dictionary key only when the list becomes empty.


## Challenge 2 Solution


In [84]:
class SearchableBiDict(BiDict):
    def has_value(self, value):
        return value in self._inverse


b = SearchableBiDict(a=1, b=2)

assert b.has_value(1)
assert not b.has_value(99)

print(b.has_value(2))


True


Because the inverse dictionary already exists, value membership is an ordinary dictionary lookup instead of a linear scan.


## Challenge 3 Solution


In [85]:
class RefreshableLazyDict(LazyDict):
    def refresh(self, key):
        value = self._loader(key)

        if value is None:
            raise KeyError(key)

        self[key] = value
        return value


counter = {"n": 0}

def changing_loader(key):
    counter["n"] += 1
    return f"{key}:{counter['n']}"


r = RefreshableLazyDict(changing_loader)

print(r["a"])
print(r["a"])
print(r.refresh("a"))
print(r["a"])


a:1
a:1
a:2
a:2


A normal lookup reuses the cached value, while `refresh` deliberately bypasses the cache and replaces it.


## Challenge 4 Solution


In [86]:
class InspectableLayeredDict(LayeredDict):
    def is_overridden(self, key):
        if key not in self:
            raise KeyError(key)
        return key in self.data


s = InspectableLayeredDict(
    {"theme": "light", "language": "en"},
    theme="dark",
)

print(s.is_overridden("theme"))
print(s.is_overridden("language"))


True
False


The visible value can come from either layer, so this method lets callers inspect the origin without exposing all implementation details.


## Challenge 5 Solution


In [87]:
class TouchExpiringDict(ExpiringDict):
    def touch(self, key):
        value = self[key]
        self.data[key] = (value, self._clock() + self._ttl)


clock = FakeClock()
t = TouchExpiringDict(5, clock)

t["x"] = 10
clock.advance(4)
t.touch("x")
clock.advance(4)

assert t["x"] == 10

clock.advance(1)
show_exception(KeyError, t.__getitem__, "x")


KeyError: 'x'


KeyError('x')

The `touch` method first performs a normal lookup, which also guarantees that an already-expired key cannot be revived accidentally.


# Review Questions

Before finishing, try answering these without running any code.

1. Why did `AliasDict` centralize canonical-key conversion in a helper?
2. Why does `MultiValueDict.getall` return a copy?
3. What extra state does `BiDict` need to maintain?
4. Why is reassignment harder in a bidirectional dictionary than initial insertion?
5. What is the role of `__missing__` in `LazyDict`?
6. Why can `get` trigger lazy loading?
7. Why was overriding only `__getitem__` insufficient for `LayeredDict`?
8. Why did `ExpiringDict` need to consider `__iter__` and `__len__`?
9. Why is an injected fake clock preferable in tests?
10. Why can a normal `update` violate a global quota only after partially mutating the mapping?
11. What is the difference between per-item validation and whole-state validation?
12. Why does `ObservableDict` notify only after the mutation succeeds?
13. Why does it iterate over a tuple of listeners?
14. Why are `area` and `perimeter` valid public keys even though they are absent from `data`?
15. When the public mapping differs from internal storage, which mapping methods should you think about?


# Summary

`UserDict` becomes especially useful when our dictionary is no longer just "a dictionary with one extra check."

In this notebook we built mappings where:

- keys can have aliases,
- one key can represent many values,
- values have an inverse lookup,
- missing values can be loaded lazily,
- visible values can come from multiple layers,
- entries can expire,
- validity can depend on the total state of the mapping,
- mutations can trigger callbacks,
- and public keys can be computed rather than stored.

The recurring question was always the same:

> What should this object mean as a mapping?

Once that is clear, we can decide which parts of the mapping protocol need to be customized so that indexing, `get`, updates, iteration, deletion, views, and conversion all behave consistently.
